<a href="https://colab.research.google.com/github/cduplan59/CFT_analysis/blob/main/ROSG_Test_Zi12345_plus_H1_perturbative_hardstress.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ROSG Test Z_i (i=1..5) — Three-case overlap admissibility audit

H0: planar control. H1: same-level overlaps only. H2: inter-level stress-test; spectral H2 is intentionally skipped beyond i=3 because it is non-admissible and topologically hub-like.

In [1]:
# ============================================
# ROSG Test Z_i (i=1..5) — Three-case overlap admissibility audit
# H0: planar 2D control, no transverse/multiplex channel
# H1: same-level admissible overlaps only (i = j)
# H2: inter-level overlaps allowed (artifact/shortcut stress test)
#
# Purpose:
#   Extend the Z_perp^(i) audit to i=4,5 while keeping H2 as a stress-test only.
#   The clean-channel verdict is evaluated on H1 separately from the H2 artifact risk.
# ============================================

import json
import math
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Dict, List, Tuple, Iterable

import numpy as np
import pandas as pd
from scipy import sparse
from scipy.sparse.linalg import eigsh, expm_multiply

try:
    import matplotlib.pyplot as plt
except Exception:
    plt = None

Node = Tuple[int, int, int, int]  # (level_i, x, y, layer)
Edge = Tuple[Node, Node, float, str]


@dataclass
class TestConfig:
    # levels to scan
    target_levels: Tuple[int, ...] = (1, 2, 3, 4, 5)
    # graph/conductance parameters
    c0: float = 1.0
    gv: float = 0.35
    w_inter: float = 0.35
    # heat trace parameters
    n_heat: int = 28
    heat_t_min: float = 1e-2
    heat_t_max: float = 1e2
    # exact eigenvalues are used below this node count, unless force_stochastic_from_i is reached
    eig_dense_limit: int = 1200
    force_stochastic_from_i: int = 4
    n_probes: int = 8
    h2_spectral_max_i: int = 3  # H2 becomes topological stress-test only beyond this level by default
    random_seed: int = 1234
    # verdict thresholds
    degree_cap_H1: int = 8
    gain_threshold: float = 0.10
    shortcut_ratio_threshold: float = 0.25
    shortcut_absolute_margin: float = 0.10
    stability_min_levels: int = 5
    # output
    out_dir: str = "/mnt/data/ROSG_Test_Zi12345_three_cases_out"


def level_size(i: int) -> int:
    """cell-4 ≡ i=0 has L=2, then L_i=2^(i+1)."""
    return 2 ** (i + 1)


def grid_nodes(i: int, layers: Iterable[int]) -> List[Node]:
    L = level_size(i)
    return [(i, x, y, layer) for layer in layers for y in range(L) for x in range(L)]


def add_edge(edges: List[Edge], u: Node, v: Node, w: float, kind: str) -> None:
    if u == v:
        return
    edges.append((u, v, float(w), kind))


def add_planar_edges(edges: List[Edge], i: int, layer: int, c: float, kind: str = "planar") -> None:
    L = level_size(i)
    for y in range(L):
        for x in range(L):
            u = (i, x, y, layer)
            if x + 1 < L:
                add_edge(edges, u, (i, x + 1, y, layer), c, kind)
            if y + 1 < L:
                add_edge(edges, u, (i, x, y + 1, layer), c, kind)


def build_case_graph(case: str, target_i: int, cfg: TestConfig) -> Tuple[List[Node], List[Edge]]:
    """Build one of the three audit graphs.

    H0: only one 2D layer at target_i.
    H1: two-layer multiplex at target_i, with vertical fiber edges only within i=j.
    H2: H1 plus parent level i=0 and deliberate inter-level shortcut edges.
    """
    i = target_i
    edges: List[Edge] = []

    if case == "H0_planar_control":
        nodes = grid_nodes(i, layers=[0])
        add_planar_edges(edges, i, layer=0, c=cfg.c0, kind="planar")
        return nodes, edges

    if case == "H1_same_level_overlaps":
        nodes = grid_nodes(i, layers=[0, 1])
        add_planar_edges(edges, i, layer=0, c=cfg.c0, kind="planar_base")
        add_planar_edges(edges, i, layer=1, c=cfg.c0, kind="planar_overlap_layer")
        L = level_size(i)
        for y in range(L):
            for x in range(L):
                add_edge(edges, (i, x, y, 0), (i, x, y, 1), cfg.gv, "same_i_fiber_overlap")
        return nodes, edges

    if case == "H2_interlevel_allowed":
        nodes = []
        nodes += grid_nodes(0, layers=[0])
        nodes += grid_nodes(i, layers=[0, 1])
        add_planar_edges(edges, 0, layer=0, c=cfg.c0, kind="planar_parent_i0")
        add_planar_edges(edges, i, layer=0, c=cfg.c0, kind="planar_base")
        add_planar_edges(edges, i, layer=1, c=cfg.c0, kind="planar_overlap_layer")
        L = level_size(i)
        L0 = level_size(0)
        scale = max(1, L // L0)
        for y in range(L):
            for x in range(L):
                add_edge(edges, (i, x, y, 0), (i, x, y, 1), cfg.gv, "same_i_fiber_overlap")
                px = min(x // scale, L0 - 1)
                py = min(y // scale, L0 - 1)
                # Deliberate stress-test shortcuts: these are NOT admissible as overlaps in H1.
                add_edge(edges, (i, x, y, 0), (0, px, py, 0), cfg.w_inter, "interlevel_shortcut")
        return nodes, edges

    raise ValueError(f"Unknown case: {case}")


def laplacian_from_edges(nodes: List[Node], edges: List[Edge]) -> Tuple[sparse.csr_matrix, Dict[str, int], Dict[str, float]]:
    idx = {n: k for k, n in enumerate(nodes)}
    n = len(nodes)
    rows, cols, data = [], [], []
    degrees = np.zeros(n)
    kind_counts: Dict[str, int] = {}
    kind_weight: Dict[str, float] = {}

    for u, v, w, kind in edges:
        if u not in idx or v not in idx:
            continue
        a, b = idx[u], idx[v]
        degrees[a] += w
        degrees[b] += w
        rows.extend([a, b])
        cols.extend([b, a])
        data.extend([-w, -w])
        kind_counts[kind] = kind_counts.get(kind, 0) + 1
        kind_weight[kind] = kind_weight.get(kind, 0.0) + w

    rows.extend(range(n))
    cols.extend(range(n))
    data.extend(degrees.tolist())
    L = sparse.coo_matrix((data, (rows, cols)), shape=(n, n)).tocsr()
    return L, kind_counts, kind_weight


def graph_diagnostics(nodes: List[Node], edges: List[Edge], kind_counts: Dict[str, int]) -> Dict[str, float]:
    deg_unweighted = np.zeros(len(nodes), dtype=int)
    idx = {n: k for k, n in enumerate(nodes)}
    for u, v, _, _ in edges:
        if u in idx and v in idx:
            deg_unweighted[idx[u]] += 1
            deg_unweighted[idx[v]] += 1
    interlevel = sum(c for k, c in kind_counts.items() if "interlevel" in k)
    same_i = sum(c for k, c in kind_counts.items() if "same_i" in k)
    return {
        "n_nodes": int(len(nodes)),
        "n_edges": int(len(edges)),
        "max_degree_unweighted": int(deg_unweighted.max()) if len(deg_unweighted) else 0,
        "mean_degree_unweighted": float(deg_unweighted.mean()) if len(deg_unweighted) else 0.0,
        "interlevel_edge_count": int(interlevel),
        "same_i_overlap_edge_count": int(same_i),
        "interlevel_edge_ratio": float(interlevel / max(1, len(edges))),
    }


def lambda1_from_eigenvalues(vals: np.ndarray, target_i: int) -> Tuple[float, float]:
    vals = np.sort(np.clip(np.real(vals), 0, None))
    positive = vals[vals > 1e-9]
    lam1 = float(positive[0]) if len(positive) else 0.0
    lam1_scaled = (4 ** target_i) * lam1
    return lam1, lam1_scaled


def heat_trace_from_eigenvalues(vals: np.ndarray, t_grid: np.ndarray) -> np.ndarray:
    vals = np.clip(np.real(vals), 0, None)
    return np.array([np.mean(np.exp(-t * vals)) for t in t_grid])


def heat_trace_exact(L: sparse.csr_matrix, t_grid: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    vals = np.linalg.eigvalsh(L.toarray())
    return vals, heat_trace_from_eigenvalues(vals, t_grid)


def grid_laplacian_eigenvalues(L: int) -> np.ndarray:
    """Exact eigenvalues of the unweighted LxL rectangular grid Laplacian with free boundary."""
    m = np.arange(L, dtype=float)
    one_d = 2.0 - 2.0 * np.cos(np.pi * m / L)
    return (one_d[:, None] + one_d[None, :]).ravel()


def heat_trace_formula_H0_H1(case: str, target_i: int, cfg: TestConfig, t_grid: np.ndarray):
    """Closed-form spectrum for H0 and H1.

    H0 is a single LxL planar grid.
    H1 is two identical layers coupled by same-site fibers of weight gv.
    The layer coupling splits the spectrum into symmetric modes lambda and
    antisymmetric modes lambda + 2*gv. This also explains why lambda1_scaled
    remains identical for H0 and H1 when gv is not smaller than the planar gap.
    """
    L = level_size(target_i)
    vals0 = grid_laplacian_eigenvalues(L)
    if case == "H0_planar_control":
        vals = vals0
    elif case == "H1_same_level_overlaps":
        vals = np.concatenate([vals0, vals0 + 2.0 * cfg.gv])
    else:
        raise ValueError(case)
    P = heat_trace_from_eigenvalues(vals, t_grid)
    lam1, lam1_scaled = lambda1_from_eigenvalues(vals, target_i)
    return vals, P, lam1, lam1_scaled



def heat_trace_hutchinson_batched(L: sparse.csr_matrix, t_grid: np.ndarray, n_probes: int, seed: int) -> np.ndarray:
    """Hutchinson trace estimate with batched Rademacher probes.

    For each heat time, compute exp(-tL) applied to an n x R matrix of probes.
    This is much faster than looping probe-by-probe for i=4,5.
    """
    rng = np.random.default_rng(seed)
    n = L.shape[0]
    V = rng.choice([-1.0, 1.0], size=(n, n_probes))
    acc = np.zeros_like(t_grid, dtype=float)
    for j, t in enumerate(t_grid):
        Y = expm_multiply((-t) * L, V)
        acc[j] = float(np.sum(V * Y) / (n * n_probes))
    return acc


def spectral_dimension_from_trace(t_grid: np.ndarray, P: np.ndarray) -> np.ndarray:
    eps = 1e-300
    logt = np.log(t_grid)
    logP = np.log(np.clip(P, eps, None))
    return -2.0 * np.gradient(logP, logt)


def thermal_window_summary(t_grid: np.ndarray, ds: np.ndarray) -> Dict[str, float]:
    n = len(t_grid)
    lo = max(2, n // 4)
    hi = min(n - 2, 3 * n // 4)
    window = ds[lo:hi]
    return {
        "ds_mean_midwindow": float(np.mean(window)),
        "ds_std_midwindow": float(np.std(window)),
        "ds_max_midwindow": float(np.max(window)),
        "ds_min_midwindow": float(np.min(window)),
    }


def sparse_lambda1(L: sparse.csr_matrix, target_i: int) -> Tuple[float, float]:
    # Connected graph Laplacians have one zero eigenvalue; compute a small cluster near 0.
    k = min(8, max(2, L.shape[0] - 1))
    vals_small = eigsh(L, k=k, which="SM", return_eigenvectors=False, tol=1e-8, maxiter=20000)
    return lambda1_from_eigenvalues(vals_small, target_i)


def analyze_case(case: str, target_i: int, cfg: TestConfig) -> Tuple[Dict[str, float], pd.DataFrame]:
    nodes, edges = build_case_graph(case, target_i, cfg)
    L, kind_counts, kind_weight = laplacian_from_edges(nodes, edges)
    gdiag = graph_diagnostics(nodes, edges, kind_counts)
    t_grid = np.logspace(math.log10(cfg.heat_t_min), math.log10(cfg.heat_t_max), cfg.n_heat)

    if case == "H2_interlevel_allowed" and target_i > cfg.h2_spectral_max_i:
        # H2 has already served as artifact stress-test by i=3; beyond that it creates
        # extremely high-degree parent hubs. Keep topological diagnostics without spending
        # computation on a non-admissible spectral channel.
        P = np.full_like(t_grid, np.nan, dtype=float)
        ds = np.full_like(t_grid, np.nan, dtype=float)
        result = {
            "case": case,
            "target_i": int(target_i),
            "L_i": level_size(target_i),
            "lambda1": float("nan"),
            "lambda1_scaled_4i": float("nan"),
            "trace_estimator": "skipped_H2_topology_only_nonadmissible",
            **gdiag,
            "ds_mean_midwindow": float("nan"),
            "ds_std_midwindow": float("nan"),
            "ds_max_midwindow": float("nan"),
            "ds_min_midwindow": float("nan"),
            "edge_kind_counts_json": json.dumps(kind_counts, sort_keys=True),
            "edge_kind_weight_json": json.dumps(kind_weight, sort_keys=True),
        }
        curves = pd.DataFrame({"target_i": target_i, "case": case, "t": t_grid, "P_t": P, "ds_eff_t": ds})
        return result, curves

    if case in ("H0_planar_control", "H1_same_level_overlaps"):
        vals, P, lam1, lam1_scaled = heat_trace_formula_H0_H1(case, target_i, cfg, t_grid)
        estimator = "closed_form_grid_layer_spectrum"
    else:
        use_exact = (L.shape[0] <= cfg.eig_dense_limit) and (target_i < cfg.force_stochastic_from_i)
        if use_exact:
            vals, P = heat_trace_exact(L, t_grid)
            lam1, lam1_scaled = lambda1_from_eigenvalues(vals, target_i)
            estimator = "exact_eigenvalues"
        else:
            lam1, lam1_scaled = sparse_lambda1(L, target_i)
            P = heat_trace_hutchinson_batched(
                L, t_grid, cfg.n_probes, cfg.random_seed + 1000 * target_i + abs(hash(case)) % 997
            )
            estimator = "hutchinson_batched_expm_multiply"

    ds = spectral_dimension_from_trace(t_grid, P)
    sdiag = thermal_window_summary(t_grid, ds)
    result = {
        "case": case,
        "target_i": int(target_i),
        "L_i": level_size(target_i),
        "lambda1": lam1,
        "lambda1_scaled_4i": lam1_scaled,
        "trace_estimator": estimator,
        **gdiag,
        **sdiag,
        "edge_kind_counts_json": json.dumps(kind_counts, sort_keys=True),
        "edge_kind_weight_json": json.dumps(kind_weight, sort_keys=True),
    }
    curves = pd.DataFrame({"target_i": target_i, "case": case, "t": t_grid, "P_t": P, "ds_eff_t": ds})
    return result, curves


def per_level_diagnostics(results: pd.DataFrame, cfg: TestConfig) -> pd.DataFrame:
    rows = []
    for i, sub in results.groupby("target_i"):
        bycase = {r["case"]: r for _, r in sub.iterrows()}
        h0 = bycase["H0_planar_control"]
        h1 = bycase["H1_same_level_overlaps"]
        h2 = bycase["H2_interlevel_allowed"]
        g1 = float(h1["ds_mean_midwindow"] - h0["ds_mean_midwindow"])
        h2_ds = float(h2["ds_mean_midwindow"]) if pd.notna(h2["ds_mean_midwindow"]) else float("nan")
        g2 = float(h2_ds - h1["ds_mean_midwindow"]) if pd.notna(h2_ds) else float("nan")
        r_short = float(g2 / g1) if pd.notna(g2) and abs(g1) > 1e-12 else float("nan")
        h1_clean = int(h1["interlevel_edge_count"]) == 0 and int(h1["max_degree_unweighted"]) <= cfg.degree_cap_H1
        h2_shortcuts = int(h2["interlevel_edge_count"]) > 0
        candidate = h1_clean and g1 >= cfg.gain_threshold
        shortcut_suspect = h2_shortcuts and (
            (pd.notna(r_short) and r_short >= cfg.shortcut_ratio_threshold) or
            (pd.notna(g2) and g2 >= cfg.shortcut_absolute_margin) or
            (int(h2["max_degree_unweighted"]) > cfg.degree_cap_H1) or
            (pd.notna(g2) and (g1 < cfg.gain_threshold <= g1 + g2))
        )
        rows.append({
            "target_i": int(i),
            "G1_H1_minus_H0": g1,
            "G2_H2_minus_H1": g2,
            "R_shortcut_G2_over_G1": r_short,
            "H1_clean_same_level_only": bool(h1_clean),
            "H2_interlevel_shortcuts_present": bool(h2_shortcuts),
            "Zperp_i_candidate": bool(candidate),
            "shortcut_artifact_suspected": bool(shortcut_suspect),
            "lambda1_scaled_H0": float(h0["lambda1_scaled_4i"]),
            "lambda1_scaled_H1": float(h1["lambda1_scaled_4i"]),
            "lambda1_scaled_H2": float(h2["lambda1_scaled_4i"]),
            "lambda1_scaled_H1_minus_H0": float(h1["lambda1_scaled_4i"] - h0["lambda1_scaled_4i"]),
            "lambda1_scaled_H2_minus_H1": float(h2["lambda1_scaled_4i"] - h1["lambda1_scaled_4i"]),
            "max_degree_H0": int(h0["max_degree_unweighted"]),
            "max_degree_H1": int(h1["max_degree_unweighted"]),
            "max_degree_H2": int(h2["max_degree_unweighted"]),
            "n_nodes_H0": int(h0["n_nodes"]),
            "n_nodes_H1": int(h1["n_nodes"]),
            "n_nodes_H2": int(h2["n_nodes"]),
        })
    return pd.DataFrame(rows).sort_values("target_i").reset_index(drop=True)


def global_verdict(level_diag: pd.DataFrame, cfg: TestConfig) -> Dict[str, object]:
    n_levels = int(len(level_diag))
    n_candidates = int(level_diag["Zperp_i_candidate"].sum())
    any_shortcut_artifact = bool(level_diag["shortcut_artifact_suspected"].any())
    all_h1_clean = bool(level_diag["H1_clean_same_level_only"].all())
    min_g1 = float(level_diag["G1_H1_minus_H0"].min())
    max_r_short = float(level_diag["R_shortcut_G2_over_G1"].replace([np.inf, -np.inf], np.nan).max(skipna=True))
    mean_g1 = float(level_diag["G1_H1_minus_H0"].mean())
    max_h1_deg = int(level_diag["max_degree_H1"].max())
    max_h2_deg = int(level_diag["max_degree_H2"].max())
    max_lam_gap_h1 = float(np.abs(level_diag["lambda1_scaled_H1_minus_H0"]).max())

    h1_verdict = (
        "Zperp_i12345_H1_stable_same_level_clean"
        if n_candidates >= cfg.stability_min_levels and all_h1_clean
        else "Zperp_i12345_H1_not_stable_or_not_clean"
    )

    if h1_verdict.endswith("clean") and not any_shortcut_artifact:
        verdict = "Zperp_i12345_stable_same_level_clean_no_H2_artifact"
        interpretation = "Same-level overlaps support a clean transverse channel across i=1..5; H2 does not add critical shortcut contamination."
    elif h1_verdict.endswith("clean") and any_shortcut_artifact:
        verdict = "Zperp_i12345_H1_stable_but_H2_interlevel_artifact_risk"
        interpretation = "H1 remains clean and positive across i=1..5, while H2 confirms that inter-level overlap links generate shortcut contamination."
    elif n_candidates > 0:
        verdict = "Zperp_partial_or_nonuniform_across_i12345"
        interpretation = "The channel appears only on part of the scanned iteration range."
    else:
        verdict = "no_stable_Zperp_i12345_candidate"
        interpretation = "Same-level overlaps do not pass the gain/cleanliness criteria."

    return {
        "global_verdict": verdict,
        "H1_clean_channel_verdict": h1_verdict,
        "levels_tested": [int(x) for x in level_diag["target_i"].tolist()],
        "candidate_levels": [int(r["target_i"]) for _, r in level_diag.iterrows() if bool(r["Zperp_i_candidate"])],
        "n_candidate_levels": n_candidates,
        "n_levels": n_levels,
        "all_H1_clean_same_level_only": all_h1_clean,
        "any_shortcut_artifact_suspected": any_shortcut_artifact,
        "min_G1_H1_minus_H0": min_g1,
        "mean_G1_H1_minus_H0": mean_g1,
        "max_R_shortcut_G2_over_G1": max_r_short,
        "max_degree_H1": max_h1_deg,
        "max_degree_H2": max_h2_deg,
        "max_abs_lambda1_scaled_H1_minus_H0": max_lam_gap_h1,
        "interpretation": interpretation,
    }


def run_audit(cfg: TestConfig) -> Dict[str, object]:
    out_dir = Path(cfg.out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    cases = ["H0_planar_control", "H1_same_level_overlaps", "H2_interlevel_allowed"]
    rows = []
    curves_all = []
    for target_i in cfg.target_levels:
        for case in cases:
            print(f"[run] i={target_i} case={case}", flush=True)
            row, curves = analyze_case(case, int(target_i), cfg)
            rows.append(row)
            curves_all.append(curves)
    results = pd.DataFrame(rows)
    curves_df = pd.concat(curves_all, ignore_index=True)
    diag = per_level_diagnostics(results, cfg)
    verdict = global_verdict(diag, cfg)

    prefix = "ROSG_Test_Zi12345_three_cases"
    summary_path = out_dir / f"{prefix}_summary.csv"
    curves_path = out_dir / f"{prefix}_heat_curves.csv"
    diag_path = out_dir / f"{prefix}_level_diagnostics.csv"
    report_path = out_dir / f"{prefix}_report.json"
    results.to_csv(summary_path, index=False)
    curves_df.to_csv(curves_path, index=False)
    diag.to_csv(diag_path, index=False)

    report = {
        "config": asdict(cfg),
        "summary": results.to_dict(orient="records"),
        "level_diagnostics": diag.to_dict(orient="records"),
        "verdict": verdict,
        "paths": {
            "summary_csv": str(summary_path),
            "level_diagnostics_csv": str(diag_path),
            "heat_curves_csv": str(curves_path),
            "report_json": str(report_path),
        },
    }

    if plt is not None:
        fig_ds = out_dir / f"{prefix}_ds_eff.png"
        fig_gain = out_dir / f"{prefix}_gains.png"
        fig_degree = out_dir / f"{prefix}_degree_growth.png"
        for (i, case), sub in curves_df.groupby(["target_i", "case"]):
            plt.plot(sub["t"], sub["ds_eff_t"], label=f"i={i} {case}")
        plt.xscale("log")
        plt.xlabel("heat time u")
        plt.ylabel("d_s^eff(u)")
        plt.title("ROSG Test Z_i three cases — i=1..5")
        plt.legend(fontsize=6)
        plt.tight_layout()
        plt.savefig(fig_ds, dpi=180)
        plt.close()

        plt.plot(diag["target_i"], diag["G1_H1_minus_H0"], marker="o", label="G1 = H1-H0")
        plt.plot(diag["target_i"], diag["G2_H2_minus_H1"], marker="o", label="G2 = H2-H1")
        plt.plot(diag["target_i"], diag["R_shortcut_G2_over_G1"], marker="o", label="R_shortcut")
        plt.xlabel("iteration level i")
        plt.ylabel("gain / shortcut ratio")
        plt.title("Z_perp gain and shortcut diagnostics across i=1..5")
        plt.legend()
        plt.tight_layout()
        plt.savefig(fig_gain, dpi=180)
        plt.close()

        plt.plot(diag["target_i"], diag["max_degree_H0"], marker="o", label="H0 max degree")
        plt.plot(diag["target_i"], diag["max_degree_H1"], marker="o", label="H1 max degree")
        plt.plot(diag["target_i"], diag["max_degree_H2"], marker="o", label="H2 max degree")
        plt.xlabel("iteration level i")
        plt.ylabel("max unweighted degree")
        plt.title("Bounded-incidence diagnostic")
        plt.legend()
        plt.tight_layout()
        plt.savefig(fig_degree, dpi=180)
        plt.close()
        report["paths"].update({"ds_eff_png": str(fig_ds), "gains_png": str(fig_gain), "degree_growth_png": str(fig_degree)})

    report_path.write_text(json.dumps(report, indent=2), encoding="utf-8")
    return report


if __name__ == "__main__":
    cfg = TestConfig()
    report = run_audit(cfg)
    print(json.dumps(report["verdict"], indent=2))
    print("Saved outputs in:", cfg.out_dir)


[run] i=1 case=H0_planar_control
[run] i=1 case=H1_same_level_overlaps
[run] i=1 case=H2_interlevel_allowed
[run] i=2 case=H0_planar_control
[run] i=2 case=H1_same_level_overlaps
[run] i=2 case=H2_interlevel_allowed
[run] i=3 case=H0_planar_control
[run] i=3 case=H1_same_level_overlaps
[run] i=3 case=H2_interlevel_allowed
[run] i=4 case=H0_planar_control
[run] i=4 case=H1_same_level_overlaps
[run] i=4 case=H2_interlevel_allowed
[run] i=5 case=H0_planar_control
[run] i=5 case=H1_same_level_overlaps
[run] i=5 case=H2_interlevel_allowed
{
  "global_verdict": "Zperp_i12345_H1_stable_but_H2_interlevel_artifact_risk",
  "H1_clean_channel_verdict": "Zperp_i12345_H1_stable_same_level_clean",
  "levels_tested": [
    1,
    2,
    3,
    4,
    5
  ],
  "candidate_levels": [
    1,
    2,
    3,
    4,
    5
  ],
  "n_candidate_levels": 5,
  "n_levels": 5,
  "all_H1_clean_same_level_only": true,
  "any_shortcut_artifact_suspected": true,
  "min_G1_H1_minus_H0": 0.27648944566455036,
  "mean_G1_H

In [2]:
cfg = TestConfig()
report = run_audit(cfg)
report["verdict"]

[run] i=1 case=H0_planar_control
[run] i=1 case=H1_same_level_overlaps
[run] i=1 case=H2_interlevel_allowed
[run] i=2 case=H0_planar_control
[run] i=2 case=H1_same_level_overlaps
[run] i=2 case=H2_interlevel_allowed
[run] i=3 case=H0_planar_control
[run] i=3 case=H1_same_level_overlaps
[run] i=3 case=H2_interlevel_allowed
[run] i=4 case=H0_planar_control
[run] i=4 case=H1_same_level_overlaps
[run] i=4 case=H2_interlevel_allowed
[run] i=5 case=H0_planar_control
[run] i=5 case=H1_same_level_overlaps
[run] i=5 case=H2_interlevel_allowed


{'global_verdict': 'Zperp_i12345_H1_stable_but_H2_interlevel_artifact_risk',
 'H1_clean_channel_verdict': 'Zperp_i12345_H1_stable_same_level_clean',
 'levels_tested': [1, 2, 3, 4, 5],
 'candidate_levels': [1, 2, 3, 4, 5],
 'n_candidate_levels': 5,
 'n_levels': 5,
 'all_H1_clean_same_level_only': True,
 'any_shortcut_artifact_suspected': True,
 'min_G1_H1_minus_H0': 0.27648944566455036,
 'mean_G1_H1_minus_H0': 0.27648944566455125,
 'max_R_shortcut_G2_over_G1': 0.6125041322040259,
 'max_degree_H1': 5,
 'max_degree_H2': 1026,
 'max_abs_lambda1_scaled_H1_minus_H0': 0.0,
 'interpretation': 'H1 remains clean and positive across i=1..5, while H2 confirms that inter-level overlap links generate shortcut contamination.'}

In [3]:
import pandas as pd
diag = pd.read_csv(report["paths"]["level_diagnostics_csv"])
diag

,target_i,G1_H1_minus_H0,G2_H2_minus_H1,R_shortcut_G2_over_G1,H1_clean_same_level_only,H2_interlevel_shortcuts_present,Zperp_i_candidate,shortcut_artifact_suspected,lambda1_scaled_H0,lambda1_scaled_H1,lambda1_scaled_H2,lambda1_scaled_H1_minus_H0,lambda1_scaled_H2_minus_H1,max_degree_H0,max_degree_H1,max_degree_H2,n_nodes_H0,n_nodes_H1,n_nodes_H2
0,1,0.276489,0.043543,0.157483,True,True,True,False,2.343146,2.343146,2.376114,0.0,0.032968,4,5,6,16,32,36
1,2,0.276489,0.066049,0.238885,True,True,True,True,2.435855,2.435855,3.362381,0.0,0.926527,4,5,18,64,128,132
2,3,0.276489,0.169351,0.612504,True,True,True,True,2.459484,2.459484,4.677449,0.0,2.217965,4,5,66,256,512,516
3,4,0.276489,NaN,NaN,True,True,True,True,2.465420,2.465420,NaN,0.0,NaN,4,5,258,1024,2048,2052
4,5,0.276489,NaN,NaN,True,True,True,True,2.466906,2.466906,NaN,0.0,NaN,4,5,1026,4096,8192,8196


In [4]:
summary = pd.read_csv(report["paths"]["summary_csv"])
summary

,case,target_i,L_i,lambda1,lambda1_scaled_4i,trace_estimator,n_nodes,n_edges,max_degree_unweighted,mean_degree_unweighted,interlevel_edge_count,same_i_overlap_edge_count,interlevel_edge_ratio,ds_mean_midwindow,ds_std_midwindow,ds_max_midwindow,ds_min_midwindow,edge_kind_counts_json,edge_kind_weight_json
0,H0_planar_control,1,4,0.585786,2.343146,closed_form_grid_layer_spectrum,16,24,4,3.000000,0,0,0.000000,1.048362,0.473527,1.653597,0.123607,"{""planar"": 24}","{""planar"": 24.0}"
1,H1_same_level_overlaps,1,4,0.585786,2.343146,closed_form_grid_layer_spectrum,32,64,5,4.000000,0,16,0.000000,1.324852,0.613214,2.074052,0.153469,"{""planar_base"": 24, ""planar_overlap_layer"": 24...","{""planar_base"": 24.0, ""planar_overlap_layer"": ..."
2,H2_interlevel_allowed,1,4,0.594029,2.376114,exact_eigenvalues,36,84,6,4.666667,16,16,0.190476,1.368394,0.659381,2.208195,0.130508,"{""interlevel_shortcut"": 16, ""planar_base"": 24,...","{""interlevel_shortcut"": 5.599999999999999, ""pl..."
3,H0_planar_control,2,8,0.152241,2.435855,closed_form_grid_layer_spectrum,64,112,4,3.500000,0,0,0.000000,1.463207,0.375194,1.973931,0.680074,"{""planar"": 112}","{""planar"": 112.0}"
4,H1_same_level_overlaps,2,8,0.152241,2.435855,closed_form_grid_layer_spectrum,128,288,5,4.500000,0,64,0.000000,1.739696,0.520444,2.395742,0.754658,"{""planar_base"": 112, ""planar_overlap_layer"": 1...","{""planar_base"": 112.0, ""planar_overlap_layer"":..."
5,H2_interlevel_allowed,2,8,0.210149,3.362381,exact_eigenvalues,132,356,18,5.393939,64,64,0.179775,1.805745,0.567027,2.525011,0.806738,"{""interlevel_shortcut"": 64, ""planar_base"": 112...","{""interlevel_shortcut"": 22.400000000000016, ""p..."
6,H0_planar_control,3,16,0.038429,2.459484,closed_form_grid_layer_spectrum,256,480,4,3.750000,0,0,0.000000,1.655776,0.410772,2.172994,0.732560,"{""planar"": 480}","{""planar"": 480.0}"
7,H1_same_level_overlaps,3,16,0.038429,2.459484,closed_form_grid_layer_spectrum,512,1216,5,4.750000,0,256,0.000000,1.932265,0.551296,2.603375,0.807144,"{""planar_base"": 480, ""planar_overlap_layer"": 4...","{""planar_base"": 480.0, ""planar_overlap_layer"":..."
8,H2_interlevel_allowed,3,16,0.073085,4.677449,exact_eigenvalues,516,1476,66,5.720930,256,256,0.173442,2.101616,0.622428,2.840550,0.847775,"{""interlevel_shortcut"": 256, ""planar_base"": 48...","{""interlevel_shortcut"": 89.59999999999977, ""pl..."
9,H0_planar_control,4,32,0.009631,2.465420,closed_form_grid_layer_spectrum,1024,1984,4,3.875000,0,0,0.000000,1.770955,0.439070,2.285239,0.759362,"{""planar"": 1984}","{""planar"": 1984.0}"


## Extension perturbative H1 — stabilité du canal same-level

Cette section prolonge `ROSG_Test_Zi12345_three_cases` sans modifier les cellules précédentes. Elle conserve le verdict central du test trois cas — `H1` comme canal admissible à profondeur égale et `H2` comme stress-test non admissible — puis teste la stabilité de `H1` sous perturbations positives bornées des conductances de fibre same-i.

Les perturbations sont appliquées aux poids de fibre `same_i_fiber_overlap` uniquement : la topologie reste inchangée, donc le degré maximal reste contrôlé. Le diagnostic vérifie que le gain `G1 = d_s(H1_pert) - d_s(H0)` reste positif, que `lambda1_scaled` ne dérive pas, et que l’incidence bornée demeure respectée.

In [5]:
# ============================================
# H1 perturbative stability extension
# Continuity with ROSG_Test_Zi12345_three_cases:
#   - keeps H0/H1 clean same-level framework
#   - freezes inter-level H2 as already non-admissible stress-test
#   - perturbs only same-i fiber conductances in H1
#   - checks whether Z_perp^(i) survives bounded conductance noise
# ============================================

from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Tuple, Dict, List
import json
import math
import numpy as np
import pandas as pd
from scipy import sparse
from scipy.sparse.linalg import expm_multiply


@dataclass
class PerturbConfig:
    target_levels: Tuple[int, ...] = (1, 2, 3, 4, 5)
    sigma_fiber_values: Tuple[float, ...] = (0.02, 0.05, 0.10, 0.20)
    n_seeds: int = 6
    gv: float = 0.35
    n_heat: int = 28
    heat_t_min: float = 1e-2
    heat_t_max: float = 1e2
    exact_anti_limit: int = 300
    n_probes: int = 10
    random_seed: int = 4321
    gain_threshold: float = 0.10
    max_allowed_degree_H1: int = 5
    gap_tol_scaled: float = 1e-8
    median_abs_rel_drift_cap_weak: float = 0.15
    median_abs_rel_drift_cap_strong: float = 0.30
    out_dir: str = "/mnt/data/ROSG_Test_Zi12345_H1_perturbative_out"


def grid_laplacian_sparse(Lside: int, c: float = 1.0) -> sparse.csr_matrix:
    n = Lside * Lside
    rows, cols, data = [], [], []
    deg = np.zeros(n, dtype=float)
    def idx(x, y):
        return y * Lside + x
    for y in range(Lside):
        for x in range(Lside):
            u = idx(x, y)
            if x + 1 < Lside:
                v = idx(x + 1, y)
                deg[u] += c; deg[v] += c
                rows += [u, v]; cols += [v, u]; data += [-c, -c]
            if y + 1 < Lside:
                v = idx(x, y + 1)
                deg[u] += c; deg[v] += c
                rows += [u, v]; cols += [v, u]; data += [-c, -c]
    rows.extend(range(n)); cols.extend(range(n)); data.extend(deg.tolist())
    return sparse.coo_matrix((data, (rows, cols)), shape=(n, n)).tocsr()


def perturb_fiber_weights(n: int, gv: float, sigma: float, rng: np.random.Generator) -> np.ndarray:
    """Positive bounded log-normal jitter around gv.

    The mean is approximately gv before clipping. Clipping prevents rare extreme
    weights from turning the perturbation test itself into a hub-like stress-test.
    Topology is unchanged, so the unweighted degree cap remains exactly 5.
    """
    if sigma <= 0:
        return np.full(n, gv, dtype=float)
    raw = gv * np.exp(sigma * rng.standard_normal(n) - 0.5 * sigma * sigma)
    lo = gv * max(0.10, 1.0 - 4.0 * sigma)
    hi = gv * (1.0 + 4.0 * sigma)
    return np.clip(raw, lo, hi)


def hutchinson_trace_sparse(A: sparse.csr_matrix, t_grid: np.ndarray, n_probes: int, seed: int) -> np.ndarray:
    rng = np.random.default_rng(seed)
    n = A.shape[0]
    V = rng.choice([-1.0, 1.0], size=(n, n_probes))
    P = np.zeros_like(t_grid, dtype=float)
    for k, t in enumerate(t_grid):
        Y = expm_multiply((-t) * A, V)
        P[k] = float(np.sum(V * Y) / (n * n_probes))
    return P


def anti_branch_trace(i: int, D_fiber: np.ndarray, t_grid: np.ndarray, pcfg: PerturbConfig) -> Tuple[np.ndarray, float, str]:
    """Trace of the antisymmetric branch L_grid + 2D.

    For two identical planar layers coupled by same-site fiber weights D(x),
    the symmetric sector is exactly the planar grid and the antisymmetric sector
    is L_grid + 2D. This avoids building the full 2N x 2N H1 matrix.
    """
    Lside = level_size(i)
    L0 = grid_laplacian_sparse(Lside, c=1.0)
    A = L0 + sparse.diags(2.0 * D_fiber, format="csr")
    n = A.shape[0]
    if n <= pcfg.exact_anti_limit:
        vals = np.linalg.eigvalsh(A.toarray())
        P = heat_trace_from_eigenvalues(vals, t_grid)
        lam_min = float(np.min(vals))
        estimator = "exact_antisymmetric_branch"
    else:
        P = hutchinson_trace_sparse(A, t_grid, pcfg.n_probes, seed=pcfg.random_seed + 10007 * i + n)
        # A sufficient guard: min eig(L0+2D) >= 2 min(D). If this lower bound is
        # already above the planar gap, then the H1 first non-zero eigenvalue is
        # certainly the same as H0. This is the expected regime for i >= 4,5.
        lam_min = float(2.0 * np.min(D_fiber))
        estimator = "hutchinson_antisymmetric_branch_with_gap_guard"
    return P, lam_min, estimator


def clean_reference_gain(i: int, pcfg: PerturbConfig) -> Dict[str, float]:
    t_grid = np.logspace(math.log10(pcfg.heat_t_min), math.log10(pcfg.heat_t_max), pcfg.n_heat)
    vals0 = grid_laplacian_eigenvalues(level_size(i))
    P0 = heat_trace_from_eigenvalues(vals0, t_grid)
    Panti = heat_trace_from_eigenvalues(vals0 + 2.0 * pcfg.gv, t_grid)
    P1 = 0.5 * (P0 + Panti)
    ds0 = spectral_dimension_from_trace(t_grid, P0)
    ds1 = spectral_dimension_from_trace(t_grid, P1)
    s0 = thermal_window_summary(t_grid, ds0)
    s1 = thermal_window_summary(t_grid, ds1)
    return {
        "target_i": int(i),
        "ds_H0_clean": s0["ds_mean_midwindow"],
        "ds_H1_clean": s1["ds_mean_midwindow"],
        "G1_clean": s1["ds_mean_midwindow"] - s0["ds_mean_midwindow"],
        "lambda1_scaled_H0_clean": lambda1_from_eigenvalues(vals0, i)[1],
    }


def analyze_H1_fiber_perturbation(i: int, sigma: float, seed_index: int, pcfg: PerturbConfig) -> Dict[str, float]:
    rng = np.random.default_rng(pcfg.random_seed + 1000 * i + 100 * seed_index + int(round(1000 * sigma)))
    t_grid = np.logspace(math.log10(pcfg.heat_t_min), math.log10(pcfg.heat_t_max), pcfg.n_heat)
    Lside = level_size(i)
    n = Lside * Lside

    vals0 = grid_laplacian_eigenvalues(Lside)
    P0 = heat_trace_from_eigenvalues(vals0, t_grid)
    ds0 = spectral_dimension_from_trace(t_grid, P0)
    s0 = thermal_window_summary(t_grid, ds0)
    lam1_grid, lam1_scaled_grid = lambda1_from_eigenvalues(vals0, i)

    D = perturb_fiber_weights(n, pcfg.gv, sigma, rng)
    Panti, anti_lam_guard, estimator = anti_branch_trace(i, D, t_grid, pcfg)
    P1 = 0.5 * (P0 + Panti)
    ds1 = spectral_dimension_from_trace(t_grid, P1)
    s1 = thermal_window_summary(t_grid, ds1)

    # H1 first non-zero eigenvalue is min(planar gap, anti branch minimum).
    # For stochastic large levels, anti_lam_guard is a lower bound; when it exceeds
    # lam1_grid, the equality with H0 is certified.
    if anti_lam_guard > lam1_grid:
        lam1_h1 = lam1_grid
        gap_guard_passed = True
    else:
        lam1_h1 = min(lam1_grid, anti_lam_guard)
        gap_guard_passed = False
    lam1_scaled_h1 = (4 ** i) * lam1_h1

    ref = clean_reference_gain(i, pcfg)
    g1 = s1["ds_mean_midwindow"] - s0["ds_mean_midwindow"]
    rel_drift = (g1 - ref["G1_clean"]) / ref["G1_clean"] if abs(ref["G1_clean"]) > 1e-12 else float("nan")

    return {
        "target_i": int(i),
        "sigma_fiber": float(sigma),
        "seed_index": int(seed_index),
        "L_i": int(Lside),
        "n_nodes_H1": int(2 * n),
        "n_edges_H1": int(2 * (2 * Lside * (Lside - 1)) + n),
        "max_degree_H1": int(5),
        "trace_estimator": estimator,
        "ds_H0_midwindow": float(s0["ds_mean_midwindow"]),
        "ds_H1_pert_midwindow": float(s1["ds_mean_midwindow"]),
        "G1_pert_H1_minus_H0": float(g1),
        "G1_clean_reference": float(ref["G1_clean"]),
        "G1_relative_drift_vs_clean": float(rel_drift),
        "G1_abs_relative_drift_vs_clean": float(abs(rel_drift)),
        "lambda1_scaled_H0": float(lam1_scaled_grid),
        "lambda1_scaled_H1_pert": float(lam1_scaled_h1),
        "lambda1_scaled_abs_drift": float(abs(lam1_scaled_h1 - lam1_scaled_grid)),
        "gap_guard_passed": bool(gap_guard_passed),
        "fiber_weight_mean": float(np.mean(D)),
        "fiber_weight_std": float(np.std(D)),
        "fiber_weight_cv": float(np.std(D) / max(1e-12, np.mean(D))),
        "fiber_weight_min": float(np.min(D)),
        "fiber_weight_max": float(np.max(D)),
    }


def aggregate_perturbation_results(rows: pd.DataFrame, pcfg: PerturbConfig) -> pd.DataFrame:
    agg = rows.groupby(["target_i", "sigma_fiber"]).agg(
        n_runs=("G1_pert_H1_minus_H0", "size"),
        G1_mean=("G1_pert_H1_minus_H0", "mean"),
        G1_std=("G1_pert_H1_minus_H0", "std"),
        G1_min=("G1_pert_H1_minus_H0", "min"),
        G1_max=("G1_pert_H1_minus_H0", "max"),
        G1_clean_reference=("G1_clean_reference", "mean"),
        rel_drift_mean=("G1_relative_drift_vs_clean", "mean"),
        rel_abs_drift_median=("G1_abs_relative_drift_vs_clean", "median"),
        rel_abs_drift_max=("G1_abs_relative_drift_vs_clean", "max"),
        positive_fraction=("G1_pert_H1_minus_H0", lambda x: float(np.mean(np.asarray(x) > pcfg.gain_threshold))),
        lambda1_scaled_drift_max=("lambda1_scaled_abs_drift", "max"),
        gap_guard_fraction=("gap_guard_passed", lambda x: float(np.mean(np.asarray(x, dtype=bool)))),
        max_degree_H1=("max_degree_H1", "max"),
        fiber_cv_mean=("fiber_weight_cv", "mean"),
        fiber_weight_min_min=("fiber_weight_min", "min"),
        fiber_weight_max_max=("fiber_weight_max", "max"),
    ).reset_index()
    agg["stable_positive_gain"] = agg["positive_fraction"] >= 0.95
    agg["bounded_incidence_ok"] = agg["max_degree_H1"] <= pcfg.max_allowed_degree_H1
    agg["gap_preserved_ok"] = agg["lambda1_scaled_drift_max"] <= pcfg.gap_tol_scaled
    agg["drift_cap"] = np.where(
        agg["sigma_fiber"] <= 0.10,
        pcfg.median_abs_rel_drift_cap_weak,
        pcfg.median_abs_rel_drift_cap_strong,
    )
    agg["relative_drift_ok"] = agg["rel_abs_drift_median"] <= agg["drift_cap"]
    agg["H1_perturbation_stable"] = (
        agg["stable_positive_gain"] &
        agg["bounded_incidence_ok"] &
        agg["gap_preserved_ok"] &
        agg["relative_drift_ok"]
    )
    return agg


def perturbation_verdict(agg: pd.DataFrame, rows: pd.DataFrame, pcfg: PerturbConfig) -> Dict[str, object]:
    all_stable = bool(agg["H1_perturbation_stable"].all())
    weak = agg[agg["sigma_fiber"] <= 0.10]
    strong = agg[agg["sigma_fiber"] > 0.10]
    weak_stable = bool(weak["H1_perturbation_stable"].all()) if len(weak) else True
    strong_stable = bool(strong["H1_perturbation_stable"].all()) if len(strong) else True
    min_g1 = float(rows["G1_pert_H1_minus_H0"].min())
    max_rel_abs = float(rows["G1_abs_relative_drift_vs_clean"].max())
    median_rel_abs = float(rows["G1_abs_relative_drift_vs_clean"].median())
    max_gap_drift = float(rows["lambda1_scaled_abs_drift"].max())
    max_degree = int(rows["max_degree_H1"].max())
    if all_stable:
        verdict = "Zperp_i12345_H1_stable_under_fiber_conductance_perturbations"
        interpretation = "The same-level H1 channel remains positive, gap-preserving, and bounded under all tested fiber-conductance perturbations."
    elif weak_stable and not strong_stable:
        verdict = "Zperp_i12345_H1_stable_under_weak_moderate_perturbations_strong_noise_sensitive"
        interpretation = "The same-level H1 channel is stable under weak/moderate perturbations, but strong fiber noise violates at least one drift criterion."
    else:
        verdict = "Zperp_i12345_H1_not_robust_under_tested_perturbations"
        interpretation = "At least one weak/moderate perturbation condition destabilizes the H1 channel diagnostic."
    return {
        "perturbation_verdict": verdict,
        "levels_tested": [int(x) for x in pcfg.target_levels],
        "sigma_fiber_values": [float(x) for x in pcfg.sigma_fiber_values],
        "n_seeds": int(pcfg.n_seeds),
        "n_runs": int(len(rows)),
        "all_aggregate_conditions_stable": all_stable,
        "weak_moderate_conditions_stable": weak_stable,
        "strong_conditions_stable": strong_stable,
        "min_G1_pert_H1_minus_H0": min_g1,
        "median_abs_relative_gain_drift": median_rel_abs,
        "max_abs_relative_gain_drift": max_rel_abs,
        "max_lambda1_scaled_abs_drift": max_gap_drift,
        "max_degree_H1": max_degree,
        "interpretation": interpretation,
    }


def run_H1_perturbation_audit(pcfg: PerturbConfig) -> Dict[str, object]:
    out_dir = Path(pcfg.out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    rows = []
    for i in pcfg.target_levels:
        for sigma in pcfg.sigma_fiber_values:
            for seed_index in range(pcfg.n_seeds):
                print(f"[perturb] i={i} sigma={sigma:.3f} seed={seed_index}", flush=True)
                rows.append(analyze_H1_fiber_perturbation(int(i), float(sigma), int(seed_index), pcfg))
    rows_df = pd.DataFrame(rows)
    agg_df = aggregate_perturbation_results(rows_df, pcfg)
    verdict = perturbation_verdict(agg_df, rows_df, pcfg)

    prefix = "ROSG_Test_Zi12345_H1_perturbative"
    rows_path = out_dir / f"{prefix}_runs.csv"
    agg_path = out_dir / f"{prefix}_aggregate.csv"
    report_path = out_dir / f"{prefix}_report.json"
    rows_df.to_csv(rows_path, index=False)
    agg_df.to_csv(agg_path, index=False)

    report = {
        "config": asdict(pcfg),
        "verdict": verdict,
        "aggregate": agg_df.to_dict(orient="records"),
        "paths": {
            "runs_csv": str(rows_path),
            "aggregate_csv": str(agg_path),
            "report_json": str(report_path),
        },
    }

    if plt is not None:
        fig_gain = out_dir / f"{prefix}_gain_stability.png"
        fig_drift = out_dir / f"{prefix}_relative_drift.png"
        fig_cv = out_dir / f"{prefix}_fiber_cv.png"

        for sigma, sub in agg_df.groupby("sigma_fiber"):
            plt.errorbar(sub["target_i"], sub["G1_mean"], yerr=sub["G1_std"].fillna(0), marker="o", label=f"sigma={sigma}")
        plt.axhline(pcfg.gain_threshold, linestyle="--", linewidth=1)
        plt.xlabel("iteration level i")
        plt.ylabel("G1 = ds(H1 perturbed) - ds(H0)")
        plt.title("H1 perturbative gain stability")
        plt.legend()
        plt.tight_layout()
        plt.savefig(fig_gain, dpi=180)
        plt.close()

        for sigma, sub in agg_df.groupby("sigma_fiber"):
            plt.plot(sub["target_i"], sub["rel_abs_drift_median"], marker="o", label=f"sigma={sigma}")
        plt.axhline(pcfg.median_abs_rel_drift_cap_weak, linestyle="--", linewidth=1)
        plt.axhline(pcfg.median_abs_rel_drift_cap_strong, linestyle=":", linewidth=1)
        plt.xlabel("iteration level i")
        plt.ylabel("median |relative drift vs clean H1 gain|")
        plt.title("H1 perturbative relative drift")
        plt.legend()
        plt.tight_layout()
        plt.savefig(fig_drift, dpi=180)
        plt.close()

        for sigma, sub in agg_df.groupby("sigma_fiber"):
            plt.plot(sub["target_i"], sub["fiber_cv_mean"], marker="o", label=f"sigma={sigma}")
        plt.xlabel("iteration level i")
        plt.ylabel("mean fiber coefficient of variation")
        plt.title("Injected fiber-conductance variability")
        plt.legend()
        plt.tight_layout()
        plt.savefig(fig_cv, dpi=180)
        plt.close()
        report["paths"].update({
            "gain_stability_png": str(fig_gain),
            "relative_drift_png": str(fig_drift),
            "fiber_cv_png": str(fig_cv),
        })

    report_path.write_text(json.dumps(report, indent=2), encoding="utf-8")
    return report


In [6]:
pcfg = PerturbConfig(
    sigma_fiber_values=(0.05, 0.10, 0.20),
    n_seeds=3,
    n_probes=4,
    n_heat=20,
)
perturb_report = run_H1_perturbation_audit(pcfg)
perturb_report["verdict"]

[perturb] i=1 sigma=0.050 seed=0
[perturb] i=1 sigma=0.050 seed=1
[perturb] i=1 sigma=0.050 seed=2
[perturb] i=1 sigma=0.100 seed=0
[perturb] i=1 sigma=0.100 seed=1
[perturb] i=1 sigma=0.100 seed=2
[perturb] i=1 sigma=0.200 seed=0
[perturb] i=1 sigma=0.200 seed=1
[perturb] i=1 sigma=0.200 seed=2
[perturb] i=2 sigma=0.050 seed=0
[perturb] i=2 sigma=0.050 seed=1
[perturb] i=2 sigma=0.050 seed=2
[perturb] i=2 sigma=0.100 seed=0
[perturb] i=2 sigma=0.100 seed=1
[perturb] i=2 sigma=0.100 seed=2
[perturb] i=2 sigma=0.200 seed=0
[perturb] i=2 sigma=0.200 seed=1
[perturb] i=2 sigma=0.200 seed=2
[perturb] i=3 sigma=0.050 seed=0
[perturb] i=3 sigma=0.050 seed=1
[perturb] i=3 sigma=0.050 seed=2
[perturb] i=3 sigma=0.100 seed=0
[perturb] i=3 sigma=0.100 seed=1
[perturb] i=3 sigma=0.100 seed=2
[perturb] i=3 sigma=0.200 seed=0
[perturb] i=3 sigma=0.200 seed=1
[perturb] i=3 sigma=0.200 seed=2
[perturb] i=4 sigma=0.050 seed=0
[perturb] i=4 sigma=0.050 seed=1
[perturb] i=4 sigma=0.050 seed=2
[perturb] 

{'perturbation_verdict': 'Zperp_i12345_H1_stable_under_fiber_conductance_perturbations',
 'levels_tested': [1, 2, 3, 4, 5],
 'sigma_fiber_values': [0.05, 0.1, 0.2],
 'n_seeds': 3,
 'n_runs': 45,
 'all_aggregate_conditions_stable': True,
 'weak_moderate_conditions_stable': True,
 'strong_conditions_stable': True,
 'min_G1_pert_H1_minus_H0': 0.27058571719756674,
 'median_abs_relative_gain_drift': 0.0006251341733556835,
 'max_abs_relative_gain_drift': 0.007404565657255211,
 'max_lambda1_scaled_abs_drift': 0.0,
 'max_degree_H1': 5,
 'interpretation': 'The same-level H1 channel remains positive, gap-preserving, and bounded under all tested fiber-conductance perturbations.'}

In [7]:
perturb_agg = pd.read_csv(perturb_report["paths"]["aggregate_csv"])
perturb_agg

,target_i,sigma_fiber,n_runs,G1_mean,G1_std,G1_min,G1_max,G1_clean_reference,rel_drift_mean,rel_abs_drift_median,...,max_degree_H1,fiber_cv_mean,fiber_weight_min_min,fiber_weight_max_max,stable_positive_gain,bounded_incidence_ok,gap_preserved_ok,drift_cap,relative_drift_ok,H1_perturbation_stable
0,1,0.05,3,0.272673,0.000096,0.272567,0.272755,0.272604,0.000252,0.000338,...,5,0.049651,0.310723,0.382526,True,True,True,0.15,True,True
1,1,0.10,3,0.272423,0.000410,0.272037,0.272853,0.272604,-0.000665,0.000911,...,5,0.091924,0.287298,0.435592,True,True,True,0.15,True,True
2,1,0.20,3,0.272050,0.000596,0.271391,0.272550,0.272604,-0.002035,0.001454,...,5,0.173554,0.235081,0.536721,True,True,True,0.30,True,True
3,2,0.05,3,0.272616,0.000066,0.272549,0.272680,0.272604,0.000042,0.000204,...,5,0.048127,0.304476,0.396922,True,True,True,0.15,True,True
4,2,0.10,3,0.272647,0.000045,0.272602,0.272693,0.272604,0.000156,0.000149,...,5,0.099489,0.267545,0.459496,True,True,True,0.15,True,True
5,2,0.20,3,0.272520,0.000268,0.272218,0.272730,0.272604,-0.000310,0.000462,...,5,0.202188,0.202480,0.597245,True,True,True,0.30,True,True
6,3,0.05,3,0.272607,0.000008,0.272602,0.272616,0.272604,0.000011,0.000007,...,5,0.050102,0.301655,0.415651,True,True,True,0.15,True,True
7,3,0.10,3,0.272618,0.000039,0.272574,0.272647,0.272604,0.000051,0.000112,...,5,0.096082,0.234153,0.481667,True,True,True,0.15,True,True
8,3,0.20,3,0.272560,0.000091,0.272478,0.272658,0.272604,-0.000163,0.000227,...,5,0.187425,0.155091,0.630000,True,True,True,0.30,True,True
9,4,0.05,3,0.270732,0.000015,0.270715,0.270742,0.272604,-0.006868,0.006840,...,5,0.050310,0.291698,0.420000,True,True,True,0.15,True,True


In [8]:
perturb_runs = pd.read_csv(perturb_report["paths"]["runs_csv"])
perturb_runs.head()

,target_i,sigma_fiber,seed_index,L_i,n_nodes_H1,n_edges_H1,max_degree_H1,trace_estimator,ds_H0_midwindow,ds_H1_pert_midwindow,...,G1_abs_relative_drift_vs_clean,lambda1_scaled_H0,lambda1_scaled_H1_pert,lambda1_scaled_abs_drift,gap_guard_passed,fiber_weight_mean,fiber_weight_std,fiber_weight_cv,fiber_weight_min,fiber_weight_max
0,1,0.05,0,4,32,64,5,exact_antisymmetric_branch,1.034886,1.307453,...,0.000136,2.343146,2.343146,0.0,True,0.350335,0.015693,0.044796,0.312437,0.372496
1,1,0.05,1,4,32,64,5,exact_antisymmetric_branch,1.034886,1.307582,...,0.000338,2.343146,2.343146,0.0,True,0.346522,0.018358,0.052977,0.310723,0.378805
2,1,0.05,2,4,32,64,5,exact_antisymmetric_branch,1.034886,1.307641,...,0.000554,2.343146,2.343146,0.0,True,0.344669,0.017640,0.051181,0.319093,0.382526
3,1,0.10,0,4,32,64,5,exact_antisymmetric_branch,1.034886,1.307738,...,0.000911,2.343146,2.343146,0.0,True,0.341007,0.035803,0.104992,0.287298,0.432310
4,1,0.10,1,4,32,64,5,exact_antisymmetric_branch,1.034886,1.307265,...,0.000827,2.343146,2.343146,0.0,True,0.358060,0.021389,0.059736,0.294624,0.385937


### Critère de lecture

Le canal `H1` est considéré comme perturbativement stable si, pour chaque niveau `i` et chaque amplitude de bruit testée :

- le gain `G1` reste positif au-dessus du seuil ;
- `lambda1_scaled_H1` reste identique au contrôle planaire ;
- le degré maximal reste borné à 5 ;
- la dérive relative médiane du gain par rapport au cas propre reste sous le seuil choisi.

Dans cette version, `H2` n’est pas relancé comme candidat : il a déjà été identifié comme source d’artefacts inter-niveaux dans le test précédent.

## H1 hard structural stress — suppression contrôlée d’overlaps same-level

Cette section poursuit `ROSG_Test_Zi12345_plus_H1_perturbative` avec un stress plus dur : au lieu de perturber seulement les poids de conductance, on supprime aléatoirement une fraction des fibres/overlaps admissibles de `H1`.

Objectif : vérifier si le canal `Z_perp^(i)` reste positif, connecté, borné et spectralement lisible lorsque le réseau d’overlaps à profondeur égale est partiellement amputé.

In [9]:
# ============================================
# H1 hard structural stress extension
# Continuity with ROSG_Test_Zi12345_plus_H1_perturbative:
#   - keeps the clean H0/H1/H2 audit and the fiber-weight perturbation audit
#   - now removes a controlled fraction of same-i fiber overlaps in H1
#   - tests whether Z_perp^(i) survives loss of admissible overlap links
# ============================================

@dataclass
class HardStressConfig:
    target_levels: Tuple[int, ...] = (1, 2, 3, 4, 5)
    dropout_values: Tuple[float, ...] = (0.05, 0.10, 0.20, 0.35, 0.50)
    n_seeds: int = 4
    gv: float = 0.35
    n_heat: int = 20
    heat_t_min: float = 1e-2
    heat_t_max: float = 1e2
    exact_anti_limit: int = 350
    n_probes: int = 4
    random_seed: int = 24680
    gain_threshold: float = 0.10
    max_allowed_degree_H1: int = 5
    gap_tol_scaled: float = 1e-8
    connected_required: bool = True
    median_abs_rel_drift_cap_low: float = 0.20
    median_abs_rel_drift_cap_high: float = 0.45
    out_dir: str = "/mnt/data/ROSG_Test_Zi12345_H1_hard_stress_out"


def dropout_fiber_weights(n: int, gv: float, p_drop: float, rng: np.random.Generator) -> Tuple[np.ndarray, int]:
    """Randomly remove a fraction p_drop of same-level H1 fiber overlaps.

    The remaining fibers keep weight gv. If the random draw removes all fibers,
    one bridge is restored so that the two layers remain connected. This keeps the
    test focused on structural sparsification rather than a trivial disconnection.
    """
    keep = rng.random(n) >= p_drop
    if not np.any(keep):
        keep[int(rng.integers(0, n))] = True
    D = gv * keep.astype(float)
    return D, int(np.sum(keep))


def anti_branch_trace_structural(i: int, D_fiber: np.ndarray, t_grid: np.ndarray, scfg: HardStressConfig) -> Tuple[np.ndarray, float, str]:
    """Trace of antisymmetric branch L_grid + 2D under structural dropout.

    Symmetric sector remains the planar grid. The anti-symmetric sector feels the
    sparse vertical-coupling potential 2D. Its smallest eigenvalue is explicitly
    estimated because strong dropout can depress the gap at low i.
    """
    Lside = level_size(i)
    L0 = grid_laplacian_sparse(Lside, c=1.0)
    A = L0 + sparse.diags(2.0 * D_fiber, format="csr")
    n = A.shape[0]
    if n <= scfg.exact_anti_limit:
        vals = np.linalg.eigvalsh(A.toarray())
        P = heat_trace_from_eigenvalues(vals, t_grid)
        lam_min = float(np.min(vals))
        estimator = "exact_antisymmetric_branch_structural_dropout"
    else:
        # Smallest algebraic eigenvalue is inexpensive for the positive Schrödinger-type graph operator.
        from scipy.sparse.linalg import eigsh
        lam_min = float(eigsh(A, k=1, which="SA", return_eigenvectors=False, tol=1e-5, maxiter=10000)[0])
        P = hutchinson_trace_sparse(A, t_grid, scfg.n_probes, seed=scfg.random_seed + 10007 * i + n + int(np.sum(D_fiber > 0)))
        estimator = "hutchinson_antisymmetric_branch_structural_dropout"
    return P, lam_min, estimator


def clean_reference_gain_hard(i: int, scfg: HardStressConfig) -> Dict[str, float]:
    # Use the same heat grid as the hard-stress audit so comparisons are internal to this notebook section.
    t_grid = np.logspace(math.log10(scfg.heat_t_min), math.log10(scfg.heat_t_max), scfg.n_heat)
    vals0 = grid_laplacian_eigenvalues(level_size(i))
    P0 = heat_trace_from_eigenvalues(vals0, t_grid)
    Panti = heat_trace_from_eigenvalues(vals0 + 2.0 * scfg.gv, t_grid)
    P1 = 0.5 * (P0 + Panti)
    ds0 = spectral_dimension_from_trace(t_grid, P0)
    ds1 = spectral_dimension_from_trace(t_grid, P1)
    s0 = thermal_window_summary(t_grid, ds0)
    s1 = thermal_window_summary(t_grid, ds1)
    return {
        "target_i": int(i),
        "ds_H0_clean": s0["ds_mean_midwindow"],
        "ds_H1_clean": s1["ds_mean_midwindow"],
        "G1_clean": s1["ds_mean_midwindow"] - s0["ds_mean_midwindow"],
        "lambda1_scaled_H0_clean": lambda1_from_eigenvalues(vals0, i)[1],
    }


def analyze_H1_structural_dropout(i: int, p_drop: float, seed_index: int, scfg: HardStressConfig) -> Dict[str, float]:
    rng = np.random.default_rng(scfg.random_seed + 1000 * i + 100 * seed_index + int(round(1000 * p_drop)))
    t_grid = np.logspace(math.log10(scfg.heat_t_min), math.log10(scfg.heat_t_max), scfg.n_heat)
    Lside = level_size(i)
    n = Lside * Lside

    vals0 = grid_laplacian_eigenvalues(Lside)
    P0 = heat_trace_from_eigenvalues(vals0, t_grid)
    ds0 = spectral_dimension_from_trace(t_grid, P0)
    s0 = thermal_window_summary(t_grid, ds0)
    lam1_grid, lam1_scaled_grid = lambda1_from_eigenvalues(vals0, i)

    D, kept_count = dropout_fiber_weights(n, scfg.gv, p_drop, rng)
    Panti, anti_lam_min, estimator = anti_branch_trace_structural(i, D, t_grid, scfg)
    P1 = 0.5 * (P0 + Panti)
    ds1 = spectral_dimension_from_trace(t_grid, P1)
    s1 = thermal_window_summary(t_grid, ds1)

    lam1_h1 = min(lam1_grid, anti_lam_min)
    lam1_scaled_h1 = (4 ** i) * lam1_h1
    gap_preserved = abs(lam1_scaled_h1 - lam1_scaled_grid) <= scfg.gap_tol_scaled

    ref = clean_reference_gain_hard(i, scfg)
    g1 = s1["ds_mean_midwindow"] - s0["ds_mean_midwindow"]
    rel_drift = (g1 - ref["G1_clean"]) / ref["G1_clean"] if abs(ref["G1_clean"]) > 1e-12 else float("nan")
    retained_fraction = kept_count / n
    # Each layer is connected; if at least one bridge remains, the two-layer graph is connected.
    connected_ok = kept_count >= 1

    return {
        "target_i": int(i),
        "dropout_fraction": float(p_drop),
        "seed_index": int(seed_index),
        "L_i": int(Lside),
        "n_sites_per_layer": int(n),
        "n_nodes_H1": int(2 * n),
        "n_possible_fibers": int(n),
        "n_retained_fibers": int(kept_count),
        "retained_fraction": float(retained_fraction),
        "n_removed_fibers": int(n - kept_count),
        "n_edges_H1": int(2 * (2 * Lside * (Lside - 1)) + kept_count),
        "max_degree_H1": int(5),
        "mean_degree_H1": float((2.0 * (2 * (2 * Lside * (Lside - 1)) + kept_count)) / (2 * n)),
        "trace_estimator": estimator,
        "ds_H0_midwindow": float(s0["ds_mean_midwindow"]),
        "ds_H1_dropout_midwindow": float(s1["ds_mean_midwindow"]),
        "G1_dropout_H1_minus_H0": float(g1),
        "G1_clean_reference": float(ref["G1_clean"]),
        "G1_relative_drift_vs_clean": float(rel_drift),
        "G1_abs_relative_drift_vs_clean": float(abs(rel_drift)),
        "lambda1_scaled_H0": float(lam1_scaled_grid),
        "lambda1_scaled_H1_dropout": float(lam1_scaled_h1),
        "lambda1_scaled_abs_drift": float(abs(lam1_scaled_h1 - lam1_scaled_grid)),
        "anti_branch_lambda_min": float(anti_lam_min),
        "gap_preserved_ok_run": bool(gap_preserved),
        "connected_ok_run": bool(connected_ok),
    }


def aggregate_hard_stress_results(rows: pd.DataFrame, scfg: HardStressConfig) -> pd.DataFrame:
    agg = rows.groupby(["target_i", "dropout_fraction"]).agg(
        n_runs=("G1_dropout_H1_minus_H0", "size"),
        retained_fraction_mean=("retained_fraction", "mean"),
        retained_fraction_min=("retained_fraction", "min"),
        G1_mean=("G1_dropout_H1_minus_H0", "mean"),
        G1_std=("G1_dropout_H1_minus_H0", "std"),
        G1_min=("G1_dropout_H1_minus_H0", "min"),
        G1_max=("G1_dropout_H1_minus_H0", "max"),
        G1_clean_reference=("G1_clean_reference", "mean"),
        rel_drift_mean=("G1_relative_drift_vs_clean", "mean"),
        rel_abs_drift_median=("G1_abs_relative_drift_vs_clean", "median"),
        rel_abs_drift_max=("G1_abs_relative_drift_vs_clean", "max"),
        positive_fraction=("G1_dropout_H1_minus_H0", lambda x: float(np.mean(np.asarray(x) > scfg.gain_threshold))),
        lambda1_scaled_drift_max=("lambda1_scaled_abs_drift", "max"),
        gap_preserved_fraction=("gap_preserved_ok_run", lambda x: float(np.mean(np.asarray(x, dtype=bool)))),
        connected_fraction=("connected_ok_run", lambda x: float(np.mean(np.asarray(x, dtype=bool)))),
        max_degree_H1=("max_degree_H1", "max"),
        mean_degree_H1_mean=("mean_degree_H1", "mean"),
        anti_lambda_min_min=("anti_branch_lambda_min", "min"),
    ).reset_index()
    agg["stable_positive_gain"] = agg["positive_fraction"] >= 0.95
    agg["bounded_incidence_ok"] = agg["max_degree_H1"] <= scfg.max_allowed_degree_H1
    agg["connected_ok"] = agg["connected_fraction"] >= 0.95 if scfg.connected_required else True
    agg["gap_preserved_ok"] = agg["gap_preserved_fraction"] >= 0.95
    agg["drift_cap"] = np.where(
        agg["dropout_fraction"] <= 0.20,
        scfg.median_abs_rel_drift_cap_low,
        scfg.median_abs_rel_drift_cap_high,
    )
    agg["relative_drift_ok"] = agg["rel_abs_drift_median"] <= agg["drift_cap"]
    agg["H1_hard_stress_stable"] = (
        agg["stable_positive_gain"] &
        agg["bounded_incidence_ok"] &
        agg["connected_ok"] &
        agg["relative_drift_ok"]
    )
    # A stronger label also requires the low spectral gap to remain unchanged.
    agg["H1_hard_stress_gap_strict_stable"] = agg["H1_hard_stress_stable"] & agg["gap_preserved_ok"]
    return agg


def hard_stress_verdict(agg: pd.DataFrame, rows: pd.DataFrame, scfg: HardStressConfig) -> Dict[str, object]:
    all_structural_stable = bool(agg["H1_hard_stress_stable"].all())
    all_gap_strict = bool(agg["H1_hard_stress_gap_strict_stable"].all())
    mild = agg[agg["dropout_fraction"] <= 0.20]
    severe = agg[agg["dropout_fraction"] > 0.20]
    mild_stable = bool(mild["H1_hard_stress_stable"].all()) if len(mild) else True
    severe_stable = bool(severe["H1_hard_stress_stable"].all()) if len(severe) else True
    mild_gap_strict = bool(mild["H1_hard_stress_gap_strict_stable"].all()) if len(mild) else True
    min_g1 = float(rows["G1_dropout_H1_minus_H0"].min())
    max_rel_abs = float(rows["G1_abs_relative_drift_vs_clean"].max())
    median_rel_abs = float(rows["G1_abs_relative_drift_vs_clean"].median())
    max_gap_drift = float(rows["lambda1_scaled_abs_drift"].max())
    max_degree = int(rows["max_degree_H1"].max())
    min_retained = float(rows["retained_fraction"].min())

    if all_structural_stable and all_gap_strict:
        verdict = "Zperp_i12345_H1_stable_under_structural_dropout_gap_preserved"
        interpretation = "Same-level H1 channel remains positive, connected, bounded, and gap-preserving under all tested dropout levels."
    elif all_structural_stable and not all_gap_strict:
        verdict = "Zperp_i12345_H1_stable_under_structural_dropout_with_low_level_gap_sensitivity"
        interpretation = "The H1 channel remains positive, connected, and bounded under structural dropout, but strict gap preservation fails in some low-level/severe dropout cases."
    elif mild_stable:
        verdict = "Zperp_i12345_H1_stable_under_mild_structural_dropout_severe_dropout_sensitive"
        interpretation = "The H1 channel is stable under mild structural dropout, but at least one severe dropout condition violates stability criteria."
    else:
        verdict = "Zperp_i12345_H1_not_robust_under_structural_dropout"
        interpretation = "Even mild removal of same-level overlaps destabilizes at least one stability criterion."

    return {
        "hard_stress_verdict": verdict,
        "levels_tested": [int(x) for x in scfg.target_levels],
        "dropout_values": [float(x) for x in scfg.dropout_values],
        "n_seeds": int(scfg.n_seeds),
        "n_runs": int(len(rows)),
        "all_structural_conditions_stable": all_structural_stable,
        "all_gap_strict_conditions_stable": all_gap_strict,
        "mild_dropout_conditions_stable": mild_stable,
        "severe_dropout_conditions_stable": severe_stable,
        "mild_dropout_gap_strict_stable": mild_gap_strict,
        "min_G1_dropout_H1_minus_H0": min_g1,
        "median_abs_relative_gain_drift": median_rel_abs,
        "max_abs_relative_gain_drift": max_rel_abs,
        "max_lambda1_scaled_abs_drift": max_gap_drift,
        "max_degree_H1": max_degree,
        "min_retained_fraction": min_retained,
        "interpretation": interpretation,
    }


def run_H1_hard_stress_audit(scfg: HardStressConfig) -> Dict[str, object]:
    out_dir = Path(scfg.out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    rows = []
    for i in scfg.target_levels:
        for p_drop in scfg.dropout_values:
            for seed_index in range(scfg.n_seeds):
                print(f"[hard-stress] i={i} dropout={p_drop:.2f} seed={seed_index}", flush=True)
                rows.append(analyze_H1_structural_dropout(int(i), float(p_drop), int(seed_index), scfg))
    rows_df = pd.DataFrame(rows)
    agg_df = aggregate_hard_stress_results(rows_df, scfg)
    verdict = hard_stress_verdict(agg_df, rows_df, scfg)

    prefix = "ROSG_Test_Zi12345_H1_hard_stress"
    rows_path = out_dir / f"{prefix}_runs.csv"
    agg_path = out_dir / f"{prefix}_aggregate.csv"
    report_path = out_dir / f"{prefix}_report.json"
    rows_df.to_csv(rows_path, index=False)
    agg_df.to_csv(agg_path, index=False)

    report = {
        "config": asdict(scfg),
        "verdict": verdict,
        "aggregate": agg_df.to_dict(orient="records"),
        "paths": {
            "runs_csv": str(rows_path),
            "aggregate_csv": str(agg_path),
            "report_json": str(report_path),
        },
    }

    if plt is not None:
        fig_gain = out_dir / f"{prefix}_gain_stability.png"
        fig_drift = out_dir / f"{prefix}_relative_drift.png"
        fig_gap = out_dir / f"{prefix}_gap_drift.png"
        fig_retained = out_dir / f"{prefix}_retained_fraction.png"

        for p_drop, sub in agg_df.groupby("dropout_fraction"):
            plt.errorbar(sub["target_i"], sub["G1_mean"], yerr=sub["G1_std"].fillna(0), marker="o", label=f"drop={p_drop}")
        plt.axhline(scfg.gain_threshold, linestyle="--", linewidth=1)
        plt.xlabel("iteration level i")
        plt.ylabel("G1 = ds(H1 dropout) - ds(H0)")
        plt.title("H1 hard structural stress: gain stability")
        plt.legend(fontsize=7)
        plt.tight_layout()
        plt.savefig(fig_gain, dpi=180)
        plt.close()

        for p_drop, sub in agg_df.groupby("dropout_fraction"):
            plt.plot(sub["target_i"], sub["rel_abs_drift_median"], marker="o", label=f"drop={p_drop}")
        plt.axhline(scfg.median_abs_rel_drift_cap_low, linestyle="--", linewidth=1)
        plt.axhline(scfg.median_abs_rel_drift_cap_high, linestyle=":", linewidth=1)
        plt.xlabel("iteration level i")
        plt.ylabel("median |relative drift vs clean H1 gain|")
        plt.title("H1 hard structural stress: relative gain drift")
        plt.legend(fontsize=7)
        plt.tight_layout()
        plt.savefig(fig_drift, dpi=180)
        plt.close()

        for p_drop, sub in agg_df.groupby("dropout_fraction"):
            plt.plot(sub["target_i"], sub["lambda1_scaled_drift_max"], marker="o", label=f"drop={p_drop}")
        plt.xlabel("iteration level i")
        plt.ylabel("max |lambda1_scaled(H1 dropout) - H0|")
        plt.title("H1 hard structural stress: low-gap drift")
        plt.legend(fontsize=7)
        plt.tight_layout()
        plt.savefig(fig_gap, dpi=180)
        plt.close()

        for p_drop, sub in agg_df.groupby("dropout_fraction"):
            plt.plot(sub["target_i"], sub["retained_fraction_mean"], marker="o", label=f"drop={p_drop}")
        plt.xlabel("iteration level i")
        plt.ylabel("mean retained same-level fiber fraction")
        plt.title("H1 hard structural stress: retained overlap fraction")
        plt.legend(fontsize=7)
        plt.tight_layout()
        plt.savefig(fig_retained, dpi=180)
        plt.close()

        report["paths"].update({
            "gain_stability_png": str(fig_gain),
            "relative_drift_png": str(fig_drift),
            "gap_drift_png": str(fig_gap),
            "retained_fraction_png": str(fig_retained),
        })

    report_path.write_text(json.dumps(report, indent=2), encoding="utf-8")
    return report


In [10]:
scfg = HardStressConfig(
    dropout_values=(0.05, 0.10, 0.20, 0.35, 0.50),
    n_seeds=2,
    n_probes=2,
    n_heat=16,
)
hard_report = run_H1_hard_stress_audit(scfg)
print(json.dumps(hard_report["verdict"], indent=2))

[hard-stress] i=1 dropout=0.05 seed=0
[hard-stress] i=1 dropout=0.05 seed=1
[hard-stress] i=1 dropout=0.10 seed=0
[hard-stress] i=1 dropout=0.10 seed=1
[hard-stress] i=1 dropout=0.20 seed=0
[hard-stress] i=1 dropout=0.20 seed=1
[hard-stress] i=1 dropout=0.35 seed=0
[hard-stress] i=1 dropout=0.35 seed=1
[hard-stress] i=1 dropout=0.50 seed=0
[hard-stress] i=1 dropout=0.50 seed=1
[hard-stress] i=2 dropout=0.05 seed=0
[hard-stress] i=2 dropout=0.05 seed=1
[hard-stress] i=2 dropout=0.10 seed=0
[hard-stress] i=2 dropout=0.10 seed=1
[hard-stress] i=2 dropout=0.20 seed=0
[hard-stress] i=2 dropout=0.20 seed=1
[hard-stress] i=2 dropout=0.35 seed=0
[hard-stress] i=2 dropout=0.35 seed=1
[hard-stress] i=2 dropout=0.50 seed=0
[hard-stress] i=2 dropout=0.50 seed=1
[hard-stress] i=3 dropout=0.05 seed=0
[hard-stress] i=3 dropout=0.05 seed=1
[hard-stress] i=3 dropout=0.10 seed=0
[hard-stress] i=3 dropout=0.10 seed=1
[hard-stress] i=3 dropout=0.20 seed=0
[hard-stress] i=3 dropout=0.20 seed=1
[hard-stress

In [11]:
hard_agg = pd.read_csv(hard_report["paths"]["aggregate_csv"])
hard_agg

,target_i,dropout_fraction,n_runs,retained_fraction_mean,retained_fraction_min,G1_mean,G1_std,G1_min,G1_max,G1_clean_reference,...,mean_degree_H1_mean,anti_lambda_min_min,stable_positive_gain,bounded_incidence_ok,connected_ok,gap_preserved_ok,drift_cap,relative_drift_ok,H1_hard_stress_stable,H1_hard_stress_gap_strict_stable
0,1,0.05,2,0.968750,0.937500,0.269306,0.000254,0.269127,0.269486,0.269127,...,3.968750,0.619284,True,True,True,True,0.20,True,True,True
1,1,0.10,2,0.937500,0.937500,0.269518,0.000045,0.269486,0.269550,0.269127,...,3.937500,0.619284,True,True,True,True,0.20,True,True,True
2,1,0.20,2,0.875000,0.812500,0.269519,0.000044,0.269488,0.269550,0.269127,...,3.875000,0.510343,True,True,True,False,0.20,True,True,False
3,1,0.35,2,0.750000,0.687500,0.268257,0.000800,0.267691,0.268822,0.269127,...,3.750000,0.416846,True,True,True,False,0.45,True,True,False
4,1,0.50,2,0.500000,0.500000,0.259694,0.001082,0.258928,0.260459,0.269127,...,3.500000,0.298164,True,True,True,False,0.45,True,True,False
5,2,0.05,2,0.968750,0.968750,0.269347,0.000037,0.269320,0.269373,0.269127,...,4.468750,0.653617,True,True,True,True,0.20,True,True,True
6,2,0.10,2,0.875000,0.796875,0.269464,0.000004,0.269461,0.269467,0.269127,...,4.375000,0.498958,True,True,True,True,0.20,True,True,True
7,2,0.20,2,0.804688,0.765625,0.269406,0.000070,0.269357,0.269456,0.269127,...,4.304688,0.470421,True,True,True,True,0.20,True,True,True
8,2,0.35,2,0.609375,0.562500,0.265995,0.002774,0.264033,0.267957,0.269127,...,4.109375,0.341291,True,True,True,True,0.45,True,True,True
9,2,0.50,2,0.515625,0.468750,0.254349,0.003328,0.251995,0.256702,0.269127,...,4.015625,0.273028,True,True,True,True,0.45,True,True,True


In [12]:
hard_runs = pd.read_csv(hard_report["paths"]["runs_csv"])
hard_runs.head()

,target_i,dropout_fraction,seed_index,L_i,n_sites_per_layer,n_nodes_H1,n_possible_fibers,n_retained_fibers,retained_fraction,n_removed_fibers,...,G1_dropout_H1_minus_H0,G1_clean_reference,G1_relative_drift_vs_clean,G1_abs_relative_drift_vs_clean,lambda1_scaled_H0,lambda1_scaled_H1_dropout,lambda1_scaled_abs_drift,anti_branch_lambda_min,gap_preserved_ok_run,connected_ok_run
0,1,0.05,0,4,16,32,16,15,0.9375,1,...,0.269486,0.269127,1.335614e-03,1.335614e-03,2.343146,2.343146,0.0,0.619284,True,True
1,1,0.05,1,4,16,32,16,16,1.0000,0,...,0.269127,0.269127,8.250563e-16,8.250563e-16,2.343146,2.343146,0.0,0.700000,True,True
2,1,0.10,0,4,16,32,16,15,0.9375,1,...,0.269486,0.269127,1.335614e-03,1.335614e-03,2.343146,2.343146,0.0,0.619284,True,True
3,1,0.10,1,4,16,32,16,15,0.9375,1,...,0.269550,0.269127,1.571894e-03,1.571894e-03,2.343146,2.343146,0.0,0.644913,True,True
4,1,0.20,0,4,16,32,16,15,0.9375,1,...,0.269550,0.269127,1.571894e-03,1.571894e-03,2.343146,2.343146,0.0,0.644913,True,True


### Critère de lecture

Le stress structurel distingue deux niveaux de stabilité :

- **stabilité structurelle** : le gain `G1 = ds(H1_dropout) - ds(H0)` reste positif, le graphe reste connecté, le degré maximal reste borné, et la dérive relative du gain reste contrôlée ;
- **stabilité stricte du gap** : en plus des critères précédents, `lambda1_scaled` reste identique à celui de `H0`.

Une sensibilité du gap à bas niveau, en particulier à `i=1`, ne doit pas être automatiquement interprétée comme invalidante si le gain diffusif reste positif et si l’incidence reste bornée. Elle indique plutôt que la suppression d’overlaps peut affecter le bas du spectre sur les petits graphes, alors que le canal reste lisible dans la trace de chaleur.